# 📊 Notebook 03 — Baseline Models (XGBoost + LSTM-only)
> **Purpose:** Train and evaluate two baseline models to benchmark
> against the full CNN-LSTM in Notebook 04.

**Inputs:** `data/X_*_2d.npy`, `data/y_*_2d.npy`, `data/X_*_seq.npy`  
**Outputs:** `data/xgb_model.json`, `data/lstm_only_model.pt`, `data/baseline_results.csv`

---

## 3.1  Load data

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
sns.set_theme(style='darkgrid')
%matplotlib inline

X_train = np.load('data/X_train_2d.npy')
y_train = np.load('data/y_train_2d.npy')
X_val   = np.load('data/X_val_2d.npy')
y_val   = np.load('data/y_val_2d.npy')
X_test  = np.load('data/X_test_2d.npy')
y_test  = np.load('data/y_test_2d.npy')

X_train_seq = np.load('data/X_train_seq.npy')
y_train_seq = np.load('data/y_train_seq.npy')
X_val_seq   = np.load('data/X_val_seq.npy')
y_val_seq   = np.load('data/y_val_seq.npy')
X_test_seq  = np.load('data/X_test_seq.npy')
y_test_seq  = np.load('data/y_test_seq.npy')

print('Shapes:', X_train.shape, y_train.shape)

## 3.2  Baseline A — XGBoost

In [ ]:
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight('balanced', y_train)

xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
    verbosity=1
)

xgb_model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val, y_val)],
    verbose=50
)

y_pred_xgb = xgb_model.predict(X_test)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
f1_xgb  = f1_score(y_test, y_pred_xgb, average='macro')

print(f'\nXGBoost  Accuracy: {acc_xgb:.4f}  F1-macro: {f1_xgb:.4f}')
print(classification_report(y_test, y_pred_xgb,
      target_names=['DOWN','FLAT','UP']))

## 3.3  XGBoost — feature importance

In [ ]:
from utils.feature_builder import FEATURE_COLS_42

importances = xgb_model.feature_importances_
feat_df = pd.DataFrame({'feature': FEATURE_COLS_42, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feat_df['feature'][::-1], feat_df['importance'][::-1],
        color='steelblue')
ax.set_title('XGBoost — Top 20 Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('data/fig_xgb_importance.png', dpi=120, bbox_inches='tight')
plt.show()

xgb_model.save_model('data/xgb_model.json')
print('XGBoost model saved.')

## 3.4  Baseline B — LSTM-only

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

class LSTMOnly(nn.Module):
    def __init__(self, input_size=42, hidden=128, num_layers=2, n_classes=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc   = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        # x: (B, T, F)
        out, _ = self.lstm(x)     # (B, T, H)
        out = out[:, -1, :]       # last timestep
        return self.fc(out)

# Dataloaders
def make_loader(X, y, batch_size=256, shuffle=True):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train_seq, y_train_seq)
val_loader   = make_loader(X_val_seq,   y_val_seq,   shuffle=False)
test_loader  = make_loader(X_test_seq,  y_test_seq,  shuffle=False)

In [ ]:
# Training loop
def train_model(model, train_loader, val_loader, epochs=20, lr=1e-3):
    optimizer  = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)
    criterion  = nn.CrossEntropyLoss()
    history    = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                logits = model(Xb)
                val_loss += criterion(logits, yb).item()
                correct  += (logits.argmax(1) == yb).sum().item()
                total    += len(yb)
        val_acc = correct / total
        scheduler.step(val_loss)

        history['train_loss'].append(total_loss / len(train_loader))
        history['val_loss'].append(val_loss / len(val_loader))
        history['val_acc'].append(val_acc)

        print(f'Ep {epoch:3d}/{epochs} | '
              f'Train Loss {total_loss/len(train_loader):.4f} | '
              f'Val Loss {val_loss/len(val_loader):.4f} | '
              f'Val Acc {val_acc:.4f}')

    return history

lstm_model = LSTMOnly().to(DEVICE)
history_lstm = train_model(lstm_model, train_loader, val_loader, epochs=20)

In [ ]:
# Evaluate on test set
lstm_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        preds = lstm_model(Xb.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

y_pred_lstm = np.array(all_preds)
acc_lstm = accuracy_score(y_test_seq, y_pred_lstm)
f1_lstm  = f1_score(y_test_seq, y_pred_lstm, average='macro')

print(f'LSTM-only  Accuracy: {acc_lstm:.4f}  F1-macro: {f1_lstm:.4f}')
print(classification_report(y_test_seq, y_pred_lstm,
      target_names=['DOWN','FLAT','UP']))

torch.save(lstm_model.state_dict(), 'data/lstm_only_model.pt')
print('LSTM-only model saved.')

## 3.5  Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history_lstm['train_loss'], label='Train')
axes[0].plot(history_lstm['val_loss'],   label='Val')
axes[0].set_title('LSTM-only — Loss')
axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history_lstm['val_acc'], color='seagreen')
axes[1].axhline(0.60, ls='--', color='red', label='Target 60%')
axes[1].set_title('LSTM-only — Validation Accuracy')
axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout()
plt.savefig('data/fig_lstm_training.png', dpi=120, bbox_inches='tight')
plt.show()

## 3.6  Save baseline results

In [ ]:
results = pd.DataFrame([
    {'Model': 'XGBoost',   'Accuracy': acc_xgb, 'F1-macro': f1_xgb},
    {'Model': 'LSTM-only', 'Accuracy': acc_lstm, 'F1-macro': f1_lstm},
])
results.to_csv('data/baseline_results.csv', index=False)
print(results.to_string(index=False))

---
> ✅ **Baselines trained.** Proceed to `04_cnn_lstm_model.ipynb`.